# SuperCodeur — LLM Training on Colab
**Train a 100M–350M parameter code LLM from scratch on free T4 GPU.**

### Steps:
1. Mount Google Drive (to save checkpoints)
2. Clone the repo
3. Download code data from HuggingFace
4. Train tokenizer & model
5. Download trained model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive')

In [ ]:
import os, sys
REPO_DIR = '/content/drive/MyDrive/agent-supercodeur'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/YOUR_USERNAME/agent-supercodeur.git "$REPO_DIR"
else:
    print('Repo already exists, pulling latest...')
    %cd "$REPO_DIR"
    !git pull

%cd "$REPO_DIR"
sys.path.insert(0, REPO_DIR)

In [ ]:
!pip install -q torch numpy pyarrow datasets regex

# Verify GPU
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
    print(f'BF16 support: {torch.cuda.is_bf16_supported()}')

In [ ]:
import subprocess, sys

# Download code data (the-stack or hasankursun/github-code-2025)
DATA_DIR = f'{REPO_DIR}/data/raw_big'
os.makedirs(DATA_DIR, exist_ok=True)

if len(os.listdir(DATA_DIR)) < 100:
    print('Downloading code dataset...')
    !python "{REPO_DIR}/download_code.py" --output "{DATA_DIR}" --max_files 50000
else:
    print(f'Data already exists: {len(os.listdir(DATA_DIR))} files')

In [ ]:
# ═══════════════════════════════════════════════════
# CONFIGURATION — Edit this cell!
# ═══════════════════════════════════════════════════

CONFIG_NAME = '300M'  # nano | 10M | 50M | 100M | 300M | 350M
EPOCHS = 5
BATCH_SIZE = 8        # T4: 8 for 300M, 16 for 100M
SEQ_LEN = 1024        # 1024 for 100M+, 512 for 50M, 256 for nano/10M
LEARNING_RATE = 3e-4
MAX_FILES = 50000

# ── Estimated memory ───
# 300M: ~600 MB weights fp16 + ~2 GB activations → fits T4 (15 GB)
print(f'Config: {CONFIG_NAME}, Batch: {BATCH_SIZE}, Seq: {SEQ_LEN}')

In [ ]:
from model.model.config import ModelConfig
from model.model.model import SuperCodeurModel
from model.tokenizer import BPE
from model.training.trainer import Trainer
from torch.utils.data import DataLoader, TensorDataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load tokenizer
tokenizer_path = f'{REPO_DIR}/checkpoints/tokenizer'
tokenizer = BPE()
tokenizer.load(tokenizer_path)
vocab_size = tokenizer.vocab_size()
print(f'Vocab size: {vocab_size}')

# Load model
config = ModelConfig.from_preset(CONFIG_NAME, vocab_size=vocab_size)
config.max_seq_len = SEQ_LEN
print(config)

model = SuperCodeurModel(config).to(device)
model = torch.compile(model)
print(f'Params: {model.count_params():,}')

# Load data
train_data = torch.load(f'{REPO_DIR}/checkpoints/data/train.pt')
val_data = torch.load(f'{REPO_DIR}/checkpoints/data/val.pt')
print(f'Train: {len(train_data)} seq, Val: {len(val_data)} seq')

train_loader = DataLoader(
    train_data, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
    pin_memory=True, num_workers=2,
)
val_loader = DataLoader(
    val_data, batch_size=BATCH_SIZE, shuffle=False,
    pin_memory=True, num_workers=2,
)

grad_accum = max(1, 512 // BATCH_SIZE)
max_steps = len(train_loader) * EPOCHS // grad_accum

trainer = Trainer(
    model=model, device=device,
    learning_rate=LEARNING_RATE, weight_decay=0.1, grad_clip=1.0,
    grad_accum_steps=grad_accum, warmup_steps=min(200, max_steps // 10),
    max_steps=max_steps, betas=(0.9, 0.95),
    precision='bf16' if torch.cuda.is_bf16_supported() else 'fp16',
)

trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=EPOCHS,
    log_interval=max(1, len(train_loader) // 10),
    val_interval=max(1, len(train_loader)),
    save_dir=f'{REPO_DIR}/checkpoints',
)

In [ ]:
best = f'{REPO_DIR}/checkpoints/best_model.pt'
if os.path.exists(best):
    model = SuperCodeurModel(config).to(device)
    model.load_state_dict(torch.load(best, map_location=device)['model_state_dict'])

    for prompt in ['def fibonacci', 'def add', 'class Stack', 'function hello', 'import numpy']:
        inp = torch.tensor([tokenizer.encode(prompt)], device=device)
        out = model.generate(inp, max_new_tokens=80, temperature=0.6, top_k=40, top_p=0.9,
                            repetition_penalty=1.15, eos_token_id=2)
        print(f'\n>> {prompt}')
        print(tokenizer.decode(out[0].tolist()))

### Results
- Checkpoints saved in Drive: `checkpoints/best_model.pt`
- Download to your local machine for inference
- To resume training, re-run the training cell (loads latest checkpoint)